# MovieLens-100K Novelty Experiments (PopDebias + ColdBridge)

This notebook runs the novelty pipeline described in `kaggle/novelty` on MovieLens-100K using GPU-friendly BGE-M3 embeddings.

It will generate:
- `results/movielens_100k/full_results.csv`
- `results/movielens_100k/best_model.txt`
- `results/figures/*.png`

In [6]:
import importlib.util
import subprocess
import sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is None:
        pip_install(pip_name or import_name)

# Only install what is missing to reduce Kaggle startup time.
ensure_package("numpy")
ensure_package("pandas")
ensure_package("matplotlib")
ensure_package("sklearn", "scikit-learn")
ensure_package("tensorflow", "tensorflow==2.19.0")
ensure_package("torch")
ensure_package("transformers")
ensure_package("sentencepiece")

print("Dependency check complete.")

Dependency check complete.


In [ ]:
import math
import random
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.decomposition import PCA
from tensorflow import keras
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

WORK_DIR = Path("/kaggle/working/llmseqrec_ml100k_single_notebook")
DATA_DIR = WORK_DIR / "data"
EMB_DIR = WORK_DIR / "embeddings" / "movielens_100k"
RESULTS_DIR = WORK_DIR / "results" / "movielens_100k"
FIG_DIR = WORK_DIR / "results" / "figures"

for p in [DATA_DIR, EMB_DIR, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
SESSION_GAP_MINUTES = 30
MIN_SESSION_LEN = 2
VAL_FRAC = 0.1
TEST_FRAC = 0.2
SEQ_LEN = 20

NUM_EPOCHS = 4
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EMB_DIM = 64

TOP_K = 20
CANDIDATE_K = 300
ALPHAS = [0.1, 0.3, 0.5, 0.7]
TAUS = [2, 3, 5, 10, 15]
DECAYS = [0.5, 0.7, 0.8, 0.9, 1.0]
LONG_TAIL_THRESHOLD = 500

print("Using work dir:", WORK_DIR)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", gpus)

2026-04-20 13:00:33.199061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776690033.224941      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776690033.233433      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776690033.252463      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776690033.252492      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776690033.252495      55 computation_placer.cc:177] computation placer alr

Using work dir: /kaggle/working/llmseqrec_ml100k_single_notebook
TensorFlow version: 2.19.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [8]:
def download_ml100k(data_dir: Path, dataset_url: str) -> tuple[Path, Path]:
    udata_path = data_dir / "ml-100k" / "u.data"
    uitem_path = data_dir / "ml-100k" / "u.item"
    if udata_path.exists() and uitem_path.exists():
        return udata_path, uitem_path

    data_dir.mkdir(parents=True, exist_ok=True)
    archive_path = data_dir / "ml-100k.zip"
    if not archive_path.exists():
        urllib.request.urlretrieve(dataset_url, archive_path)

    with zipfile.ZipFile(archive_path, "r") as zf:
        zf.extractall(data_dir)

    if not (udata_path.exists() and uitem_path.exists()):
        raise FileNotFoundError("MovieLens-100K files not found after extraction.")
    return udata_path, uitem_path


def load_interactions(udata_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        udata_path,
        sep="\t",
        names=["UserId", "ItemId", "Rating", "Timestamp"],
        engine="python",
    )


def sessionize(interactions: pd.DataFrame, gap_minutes: int, min_session_len: int) -> pd.DataFrame:
    df = interactions.copy()
    df["Timestamp"] = df["Timestamp"].astype(int)
    df["ItemId"] = df["ItemId"].astype(int)
    df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

    gap_seconds = gap_minutes * 60
    user_changed = df["UserId"].ne(df["UserId"].shift(1))
    ts_gap = df["Timestamp"].diff().fillna(0)
    new_session = user_changed | (ts_gap > gap_seconds)
    df["SessionId"] = new_session.cumsum().astype(int)

    valid = df.groupby("SessionId").size()
    valid_ids = valid[valid >= min_session_len].index
    df = df[df["SessionId"].isin(valid_ids)].copy()

    remap = {old: i + 1 for i, old in enumerate(sorted(df["SessionId"].unique()))}
    df["SessionId"] = df["SessionId"].map(remap).astype(int)

    df["Time"] = pd.to_datetime(df["Timestamp"], unit="s", utc=True).dt.tz_localize(None)
    df["Time"] = df["Time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    df["Reward"] = df["Rating"].astype(float)
    return df[["SessionId", "ItemId", "Time", "Reward"]].sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)


def temporal_split(sessions_df: pd.DataFrame, val_frac: float, test_frac: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = sessions_df.copy()
    df["Time"] = pd.to_datetime(df["Time"])

    session_end = df.groupby("SessionId")["Time"].max().sort_values()
    session_ids = session_end.index.to_numpy()
    n = len(session_ids)

    n_test = max(1, int(round(n * test_frac)))
    n_val = max(1, int(round(n * val_frac)))
    n_train = n - n_val - n_test
    if n_train <= 0:
        raise ValueError("Split produced empty train partition.")

    train_ids = set(session_ids[:n_train])
    val_ids = set(session_ids[n_train:n_train + n_val])
    test_ids = set(session_ids[n_train + n_val:])

    train_df = df[df["SessionId"].isin(train_ids)].copy()
    val_df = df[df["SessionId"].isin(val_ids)].copy()
    test_df = df[df["SessionId"].isin(test_ids)].copy()

    return (
        train_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
        val_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
        test_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True),
    )


def build_cases(split_df: pd.DataFrame, n_withheld: int = 1) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray], dict[int, int]]:
    prompts: dict[int, np.ndarray] = {}
    gts: dict[int, np.ndarray] = {}
    lengths: dict[int, int] = {}

    for sid, cur in split_df.groupby("SessionId"):
        items = cur["ItemId"].to_numpy(dtype=int)
        if len(items) <= n_withheld:
            continue
        prompt = items[:-n_withheld]
        gt = items[-n_withheld:]
        if len(prompt) == 0:
            continue
        sid = int(sid)
        prompts[sid] = prompt
        gts[sid] = gt
        lengths[sid] = len(prompt)
    return prompts, gts, lengths


def load_item_metadata(uitem_path: Path) -> pd.DataFrame:
    genre_cols = [
        "unknown", "action", "adventure", "animation", "childrens", "comedy",
        "crime", "documentary", "drama", "fantasy", "film_noir", "horror",
        "musical", "mystery", "romance", "sci_fi", "thriller", "war", "western",
    ]
    cols = ["ItemId", "title", "release_date", "video_release_date", "imdb_url", *genre_cols]
    df = pd.read_csv(
        uitem_path,
        sep="|",
        names=cols,
        encoding="latin-1",
        usecols=list(range(len(cols))),
        engine="python",
    )

    def build_text(row: pd.Series) -> str:
        genres = [g.replace("_", " ") for g in genre_cols if int(row[g]) == 1]
        genre_text = ", ".join(genres) if genres else "unknown"
        return f"{row['title']}. Genres: {genre_text}."

    df["item_text"] = df.apply(build_text, axis=1)
    return df[["ItemId", "title", "item_text"]]


def normalize_rows(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def encode_bge_cpu(item_ids: np.ndarray, item_text_df: pd.DataFrame, output_npy: Path, batch_size: int = 32, max_length: int = 256) -> np.ndarray:
    output_npy.parent.mkdir(parents=True, exist_ok=True)

    text_lookup = item_text_df.set_index("ItemId")["item_text"].to_dict()
    texts = [text_lookup.get(int(i), f"Movie item {int(i)}.") for i in item_ids]

    if output_npy.exists():
        cached = np.load(output_npy)
        if cached.shape[0] == len(item_ids):
            return cached

    import torch
    import torch.nn.functional as F

    device = "cpu"
    tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

    try:
        model = AutoModel.from_pretrained("BAAI/bge-m3", dtype=torch.float32)
    except TypeError:
        model = AutoModel.from_pretrained("BAAI/bge-m3", torch_dtype=torch.float32)
    model.to(device)
    model.eval()

    all_chunks = []
    for start in range(0, len(texts), batch_size):
        cur = texts[start:start + batch_size]
        encoded = tokenizer(
            cur,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            out = model(**encoded)
            token_emb = out.last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).expand(token_emb.size()).float()
            pooled = (token_emb * mask).sum(dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)
            pooled = F.normalize(pooled, p=2, dim=1)

        all_chunks.append(pooled.cpu().numpy().astype(np.float32))

    embeddings = np.vstack(all_chunks).astype(np.float32)
    np.save(output_npy, embeddings)
    return embeddings


def build_id_maps(all_item_ids: np.ndarray) -> tuple[dict[int, int], dict[int, int]]:
    item_to_reduced = {int(item): idx + 1 for idx, item in enumerate(all_item_ids.tolist())}
    reduced_to_item = {idx + 1: int(item) for idx, item in enumerate(all_item_ids.tolist())}
    return item_to_reduced, reduced_to_item


def to_reduced_sequence(seq: np.ndarray, item_to_reduced: dict[int, int]) -> list[int]:
    return [item_to_reduced[int(i)] for i in seq if int(i) in item_to_reduced]


def build_train_examples(train_df: pd.DataFrame, item_to_reduced: dict[int, int], seq_len: int) -> tuple[np.ndarray, np.ndarray]:
    x_rows: list[np.ndarray] = []
    y_rows: list[int] = []

    for _, cur in train_df.groupby("SessionId"):
        items = to_reduced_sequence(cur["ItemId"].to_numpy(dtype=int), item_to_reduced)
        if len(items) < 2:
            continue
        for t in range(1, len(items)):
            prompt = items[max(0, t - seq_len):t]
            target = items[t]
            x = np.zeros(seq_len, dtype=np.int32)
            x[-len(prompt):] = np.array(prompt, dtype=np.int32)
            x_rows.append(x)
            y_rows.append(target - 1)  # logits are for item indices 1..num_items
    return np.array(x_rows, dtype=np.int32), np.array(y_rows, dtype=np.int32)


def build_predict_array(prompts: dict[int, np.ndarray], item_to_reduced: dict[int, int], seq_len: int) -> tuple[list[int], np.ndarray, list[list[int]]]:
    sids: list[int] = []
    x_rows: list[np.ndarray] = []
    seen_rows: list[list[int]] = []

    for sid, prompt in prompts.items():
        red = to_reduced_sequence(prompt, item_to_reduced)
        if not red:
            continue
        arr = np.zeros(seq_len, dtype=np.int32)
        arr[-len(red[-seq_len:]):] = np.array(red[-seq_len:], dtype=np.int32)
        sids.append(int(sid))
        x_rows.append(arr)
        seen_rows.append(red)

    if not x_rows:
        return [], np.empty((0, seq_len), dtype=np.int32), []
    return sids, np.array(x_rows, dtype=np.int32), seen_rows


def build_sasrec_like_model(num_items: int, seq_len: int, emb_dim: int, pretrained_item_embeddings: np.ndarray | None = None) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb",
    )

    x = item_emb_layer(inp)

    pos_idx = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb_layer = keras.layers.Embedding(input_dim=seq_len, output_dim=emb_dim, name="pos_emb")
    pos_emb = pos_emb_layer(pos_idx)
    x = x + pos_emb

    key_dim = max(1, emb_dim // 2)
    attn = keras.layers.MultiHeadAttention(num_heads=2, key_dim=key_dim, dropout=0.2)
    attn_out = attn(x, x, use_causal_mask=True)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    ffn = keras.Sequential(
        [
            keras.layers.Dense(emb_dim * 2, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(emb_dim),
        ]
    )
    ffn_out = ffn(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_out)

    non_zero = tf.cast(tf.not_equal(inp, 0), tf.int32)
    last_idx = tf.maximum(tf.reduce_sum(non_zero, axis=1) - 1, 0)
    batch_idx = tf.range(tf.shape(inp)[0], dtype=tf.int32)
    gather_idx = tf.stack([batch_idx, last_idx], axis=1)
    last_hidden = tf.gather_nd(x, gather_idx)

    item_table = item_emb_layer.embeddings[1:]  # [num_items, emb_dim]
    logits = tf.matmul(last_hidden, item_table, transpose_b=True)

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings is not None:
        if pretrained_item_embeddings.shape != (num_items, emb_dim):
            raise ValueError(
                f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
            )
        weights = item_emb_layer.get_weights()
        table = weights[0]
        table[1:, :] = pretrained_item_embeddings.astype(np.float32)
        item_emb_layer.set_weights([table])

    return model


def predict_topk(model: keras.Model, prompts: dict[int, np.ndarray], item_to_reduced: dict[int, int], reduced_to_item: dict[int, int], seq_len: int, top_k: int) -> dict[int, np.ndarray]:
    sids, x_arr, seen_rows = build_predict_array(prompts, item_to_reduced, seq_len)
    if len(sids) == 0:
        return {}

    logits = model.predict(x_arr, verbose=0, batch_size=1024)  # [B, num_items]
    preds: dict[int, np.ndarray] = {}

    for i, sid in enumerate(sids):
        scores = logits[i].astype(np.float64)
        seen = set(seen_rows[i])
        if seen:
            seen_idx = [r - 1 for r in seen if r > 0 and (r - 1) < len(scores)]
            scores[np.array(seen_idx, dtype=int)] = -1e12

        order = np.argsort(scores)[::-1][:top_k]
        rec_reduced = [int(o) + 1 for o in order]
        rec_orig = [reduced_to_item[r] for r in rec_reduced if r in reduced_to_item]
        preds[sid] = np.array(rec_orig, dtype=int)

    return preds


def predict_popular(prompts: dict[int, np.ndarray], train_counts: dict[int, int], top_k: int) -> dict[int, np.ndarray]:
    popular_items = [int(i) for i, _ in sorted(train_counts.items(), key=lambda kv: kv[1], reverse=True)]
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        seen = set(int(i) for i in prompt)
        recs = [i for i in popular_items if i not in seen][:top_k]
        out[int(sid)] = np.array(recs, dtype=int)
    return out


def session_embedding(prompt: np.ndarray, item_to_idx: dict[int, int], emb_norm: np.ndarray, decay: float = 1.0) -> np.ndarray | None:
    idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
    if not idxs:
        return None
    vecs = emb_norm[idxs]
    if len(vecs) == 1:
        return vecs[0]
    if decay == 1.0:
        out = vecs.mean(axis=0)
    else:
        n = len(vecs)
        weights = np.array([decay ** (n - i - 1) for i in range(n)], dtype=np.float32)
        weights = weights / np.sum(weights)
        out = np.average(vecs, axis=0, weights=weights)
    norm = np.linalg.norm(out)
    return out if norm == 0 else out / norm


def debias_weights(counts: np.ndarray, alpha: float, variant: str, ranks: np.ndarray, median_count: float) -> np.ndarray:
    c = np.maximum(counts.astype(np.float64), 1.0)
    if variant == "inverse":
        return 1.0 / np.power(c, alpha)
    if variant == "log_norm":
        return 1.0 / np.power(1.0 + np.log(c), alpha)
    if variant == "rank":
        return 1.0 / np.power(np.maximum(ranks.astype(np.float64), 1.0), alpha)
    if variant == "sigmoid":
        x = -alpha * ((c / max(median_count, 1.0)) - 1.0)
        return 1.0 / (1.0 + np.exp(-x))
    raise ValueError(f"Unknown variant: {variant}")


def rerank_popdebias(base_candidates: dict[int, np.ndarray], prompts: dict[int, np.ndarray], item_to_idx: dict[int, int], emb_norm: np.ndarray, item_counts: dict[int, int], alpha: float, top_k: int, variant: str = "inverse") -> dict[int, np.ndarray]:
    sorted_pop = sorted(item_counts.items(), key=lambda kv: kv[1], reverse=True)
    rank_map = {int(item): idx + 1 for idx, (item, _) in enumerate(sorted_pop)}
    median_count = float(np.median(np.array(list(item_counts.values()), dtype=np.float64)))

    out: dict[int, np.ndarray] = {}
    for sid, cand in base_candidates.items():
        q = session_embedding(prompts[sid], item_to_idx, emb_norm, decay=1.0)
        if q is None:
            out[sid] = np.array(cand[:top_k], dtype=int)
            continue

        cand = np.array([int(i) for i in cand if int(i) in item_to_idx], dtype=int)
        if cand.size == 0:
            out[sid] = np.array([], dtype=int)
            continue

        idxs = np.array([item_to_idx[int(i)] for i in cand], dtype=int)
        sims = emb_norm[idxs] @ q
        counts = np.array([item_counts.get(int(i), 1) for i in cand], dtype=np.float64)
        ranks = np.array([rank_map.get(int(i), len(rank_map) + 1) for i in cand], dtype=np.float64)
        weights = debias_weights(counts, alpha, variant, ranks, median_count)
        scores = sims * weights

        seen = set(int(i) for i in prompts[sid])
        if seen:
            mask = np.array([i not in seen for i in cand], dtype=bool)
            cand = cand[mask]
            scores = scores[mask]
        if cand.size == 0:
            out[sid] = np.array([], dtype=int)
            continue

        order = np.argsort(scores)[::-1]
        out[sid] = cand[order][:top_k].astype(int)
    return out


def predict_cold_branch(prompts: dict[int, np.ndarray], all_item_ids: np.ndarray, item_to_idx: dict[int, int], emb_norm: np.ndarray, top_k: int, decay: float) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        q = session_embedding(prompt, item_to_idx, emb_norm, decay=decay)
        if q is None:
            out[sid] = all_item_ids[:top_k].astype(int)
            continue
        scores = emb_norm @ q
        seen_idxs = [item_to_idx[int(i)] for i in prompt if int(i) in item_to_idx]
        if seen_idxs:
            scores[np.array(seen_idxs, dtype=int)] = -1e12
        order = np.argsort(scores)[::-1][:top_k]
        out[sid] = all_item_ids[order].astype(int)
    return out


def route_coldbridge(warm_preds: dict[int, np.ndarray], cold_preds: dict[int, np.ndarray], prompts: dict[int, np.ndarray], tau: int, top_k: int) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, prompt in prompts.items():
        out[sid] = (cold_preds[sid] if len(prompt) <= tau else warm_preds[sid])[:top_k]
    return out


def evaluate_at_k(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], k: int) -> tuple[float, float]:
    hrs = []
    ndcgs = []
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:k]]
        if target in rec:
            hrs.append(1.0)
            rank = rec.index(target) + 1
            ndcgs.append(1.0 / math.log2(rank + 1.0))
        else:
            hrs.append(0.0)
            ndcgs.append(0.0)
    return float(np.mean(ndcgs) if ndcgs else 0.0), float(np.mean(hrs) if hrs else 0.0)


def long_tail_hr10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], item_counts: dict[int, int], threshold: int) -> float:
    hits = 0
    total = 0
    for sid, gt in gts.items():
        target = int(gt[0])
        if item_counts.get(target, 0) >= threshold:
            continue
        total += 1
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        if target in rec:
            hits += 1
    return float(hits / total) if total > 0 else 0.0


def catalog_coverage10(preds: dict[int, np.ndarray], num_items: int) -> float:
    used = set()
    for rec in preds.values():
        used.update(int(i) for i in rec[:10])
    return float(len(used) / num_items) if num_items > 0 else 0.0


def serendipity10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], expected_popular_top10: list[int]) -> float:
    vals = []
    expected = set(expected_popular_top10)
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        vals.append(1.0 if (target in rec and target not in expected) else 0.0)
    return float(np.mean(vals) if vals else 0.0)


def ild10(preds: dict[int, np.ndarray], item_to_idx: dict[int, int], emb_norm: np.ndarray) -> float:
    vals = []
    for rec in preds.values():
        ids = [int(i) for i in rec[:10] if int(i) in item_to_idx]
        if len(ids) < 2:
            continue
        idxs = np.array([item_to_idx[i] for i in ids], dtype=int)
        vecs = emb_norm[idxs]
        sims = vecs @ vecs.T
        iu = np.triu_indices(len(ids), k=1)
        if len(iu[0]) == 0:
            continue
        vals.append(float(np.mean(1.0 - sims[iu])))
    return float(np.mean(vals) if vals else 0.0)


def segmented_hr10(preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], lengths: dict[int, int], cold_thr: int = 5) -> tuple[float, float]:
    cold_vals = []
    warm_vals = []
    for sid, gt in gts.items():
        target = int(gt[0])
        rec = [int(i) for i in preds.get(sid, np.array([], dtype=int))[:10]]
        hit = 1.0 if target in rec else 0.0
        if lengths.get(sid, 0) <= cold_thr:
            cold_vals.append(hit)
        else:
            warm_vals.append(hit)
    cold_hr = float(np.mean(cold_vals) if cold_vals else 0.0)
    warm_hr = float(np.mean(warm_vals) if warm_vals else 0.0)
    return cold_hr, warm_hr


def evaluate_predictions(model_name: str, preds: dict[int, np.ndarray], gts: dict[int, np.ndarray], item_counts: dict[int, int], num_items: int, emb_norm: np.ndarray, item_to_idx: dict[int, int], expected_pop_top10: list[int], alpha: float | None, tau: int | None, training_time: float, inference_time: float) -> dict[str, Any]:
    ndcg10, hr10 = evaluate_at_k(preds, gts, 10)
    ndcg20, hr20 = evaluate_at_k(preds, gts, 20)
    row = {
        "model_name": model_name,
        "alpha": alpha,
        "tau": tau,
        "NDCG@10": ndcg10,
        "NDCG@20": ndcg20,
        "HR@10": hr10,
        "HR@20": hr20,
        "LongTail_HR@10": long_tail_hr10(preds, gts, item_counts, LONG_TAIL_THRESHOLD),
        "CatalogCoverage": catalog_coverage10(preds, num_items),
        "Serendipity": serendipity10(preds, gts, expected_pop_top10),
        "ILD@10": ild10(preds, item_to_idx, emb_norm),
        "training_time_sec": float(training_time),
        "inference_time_sec": float(inference_time),
    }
    return row


def choose_best_joint(rows: list[dict[str, Any]]) -> dict[str, Any]:
    scored = []
    for row in rows:
        joint = 0.7 * float(row["NDCG@10"]) + 0.3 * float(row["LongTail_HR@10"])
        scored.append((joint, row))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def save_best_model(best_row: dict[str, Any], baseline_row: dict[str, Any] | None, output_path: Path) -> None:
    baseline_ndcg = float(baseline_row["NDCG@10"]) if baseline_row is not None else None
    best_ndcg = float(best_row["NDCG@10"])

    if baseline_ndcg and baseline_ndcg > 0:
        delta = ((best_ndcg - baseline_ndcg) / baseline_ndcg) * 100.0
        delta_text = f"{delta:+.2f}%"
        base_text = f"{baseline_ndcg:.6f}"
    else:
        delta_text = "N/A"
        base_text = "N/A"

    lines = [
        f"BEST MODEL: {best_row['model_name']}",
        f"Best Alpha: {best_row.get('alpha')}",
        f"Best Tau:   {best_row.get('tau')}",
        f"NDCG@10:    {best_ndcg:.6f} (vs baseline: {base_text}, delta: {delta_text})",
        f"HR@10:      {float(best_row['HR@10']):.6f}",
        f"LT-HR@10:   {float(best_row['LongTail_HR@10']):.6f}",
        f"ILD@10:     {float(best_row['ILD@10']):.6f}",
    ]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

In [9]:
# Keras-3-safe override for the sequence model used below.
def build_sasrec_like_model(num_items: int, seq_len: int, emb_dim: int, pretrained_item_embeddings: np.ndarray | None = None) -> keras.Model:
    inp = keras.Input(shape=(seq_len,), dtype="int32")

    item_emb_layer = keras.layers.Embedding(
        input_dim=num_items + 1,
        output_dim=emb_dim,
        mask_zero=True,
        name="item_emb",
    )
    x = item_emb_layer(inp)

    attn = keras.layers.MultiHeadAttention(num_heads=2, key_dim=max(1, emb_dim // 2), dropout=0.2)
    attn_out = attn(x, x, use_causal_mask=True)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attn_out)

    ffn = keras.Sequential(
        [
            keras.layers.Dense(emb_dim * 2, activation="relu"),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(emb_dim),
        ]
    )
    ffn_out = ffn(x)
    x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_out)

    pooled = keras.layers.GlobalAveragePooling1D()(x)
    logits = keras.layers.Dense(num_items, name="logits")(pooled)

    model = keras.Model(inputs=inp, outputs=logits)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    )

    if pretrained_item_embeddings is not None:
        if pretrained_item_embeddings.shape != (num_items, emb_dim):
            raise ValueError(
                f"Expected pretrained embeddings shape {(num_items, emb_dim)}, got {pretrained_item_embeddings.shape}"
            )
        table = item_emb_layer.get_weights()[0]
        table[1:, :] = pretrained_item_embeddings.astype(np.float32)
        item_emb_layer.set_weights([table])

    return model

In [ ]:
# v2 novelty utilities (relevance-preserving rerank + soft ColdBridge)
import itertools

RUN_MODE = "fast"  # "fast" keeps runtime practical on Kaggle; set "full" for exhaustive sweep.
HR_FLOOR = 0.07681
NDCG_FLOOR = 0.04151
COVERAGE_MIN = 0.30 if RUN_MODE == "fast" else 0.35

MAX_SWEEP_CONFIGS = 720 if RUN_MODE == "fast" else None
FAST_ALPHA_GRID = [0.01, 0.05, 0.1]
FAST_LAMBDA_POP_GRID = [0.05, 0.1, 0.15]
FAST_LAMBDA_SIM_GRID = [0.0, 0.05]
FAST_TAU_GRID = [5, 7, 10]
FAST_STEEPNESS_GRID = [1.0, 2.0]
FAST_DECAY_GRID = [0.7, 0.8, 0.9]
FAST_TOP_N_ANCHOR_GRID = [0, 1, 3]


def _patched_product_args(args: tuple[Any, ...]) -> tuple[Any, ...]:
    if RUN_MODE != "fast" or len(args) != 7:
        return args
    return (
        FAST_ALPHA_GRID,
        FAST_LAMBDA_POP_GRID,
        FAST_LAMBDA_SIM_GRID,
        FAST_TAU_GRID,
        FAST_STEEPNESS_GRID,
        FAST_DECAY_GRID,
        FAST_TOP_N_ANCHOR_GRID,
    )


def _install_fast_product_patch() -> None:
    if not hasattr(itertools, "_original_product"):
        setattr(itertools, "_original_product", itertools.product)

    original_product = getattr(itertools, "_original_product")

    def product_with_controls(*args, **kwargs):
        patched_args = _patched_product_args(args)
        if RUN_MODE == "fast" and len(args) == 7 and not getattr(product_with_controls, "_announced", False):
            total = 1
            for grid in patched_args:
                total *= len(grid)
            print(
                f"[FAST_MODE] Reduced sweep grid enabled "
                f"(max combos={total}, coverage_floor={COVERAGE_MIN:.2f}, max_trials={MAX_SWEEP_CONFIGS})."
            )
            setattr(product_with_controls, "_announced", True)

        iterator = original_product(*patched_args, **kwargs)
        if MAX_SWEEP_CONFIGS is None:
            for combo in iterator:
                yield combo
            return

        for i, combo in enumerate(iterator, start=1):
            if i > MAX_SWEEP_CONFIGS:
                print(f"[EARLY_STOP] Reached MAX_SWEEP_CONFIGS={MAX_SWEEP_CONFIGS}.")
                break
            yield combo

    if getattr(itertools.product, "__name__", "") != "product_with_controls":
        itertools.product = product_with_controls


_install_fast_product_patch()

CORE_TARGETS = {
    "LLM2SASRec": {"NDCG@10": 0.05994, "HR@10": 0.12472, "CatalogCoverage": 0.23228},
    "BERT4Rec": {"NDCG@10": 0.05594, "HR@10": 0.10913, "CatalogCoverage": 0.24300},
    "SASRec": {"NDCG@10": 0.07303, "HR@10": 0.14254, "CatalogCoverage": 0.36152},
}


def z_score_normalize(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float64)
    std = arr.std()
    if std < 1e-9:
        return np.zeros_like(arr, dtype=np.float64)
    return (arr - arr.mean()) / std


def predict_topk_with_scores(
    model: keras.Model,
    prompts: dict[int, np.ndarray],
    item_to_reduced: dict[int, int],
    reduced_to_item: dict[int, int],
    seq_len: int,
    top_k: int,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    sids, x_arr, seen_rows = build_predict_array(prompts, item_to_reduced, seq_len)
    if len(sids) == 0:
        return {}, {}

    logits = model.predict(x_arr, verbose=0, batch_size=1024)
    out_items: dict[int, np.ndarray] = {}
    out_scores: dict[int, np.ndarray] = {}

    for i, sid in enumerate(sids):
        scores = logits[i].astype(np.float64)
        seen = set(seen_rows[i])
        if seen:
            seen_idx = [r - 1 for r in seen if r > 0 and (r - 1) < len(scores)]
            if seen_idx:
                scores[np.array(seen_idx, dtype=int)] = -1e12

        order = np.argsort(scores)[::-1][:top_k]
        reduced_items = np.array([int(o) + 1 for o in order], dtype=int)
        orig_items = np.array([reduced_to_item[r] for r in reduced_items if r in reduced_to_item], dtype=int)
        out_items[sid] = orig_items
        out_scores[sid] = scores[order][: len(orig_items)]

    return out_items, out_scores


def topk_from_candidates_scores(
    candidates: dict[int, np.ndarray], scores: dict[int, np.ndarray], top_k: int
) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, items in candidates.items():
        cur_items = np.asarray(items, dtype=int)
        cur_scores = np.asarray(scores.get(sid, np.zeros(len(cur_items), dtype=np.float64)), dtype=np.float64)
        if cur_items.size == 0:
            out[sid] = np.array([], dtype=int)
            continue
        order = np.argsort(cur_scores)[::-1][:top_k]
        out[sid] = cur_items[order].astype(int)
    return out


def rerank_popdebias_v2(
    candidates: dict[int, np.ndarray],
    warm_scores: dict[int, np.ndarray],
    prompts: dict[int, np.ndarray],
    item_to_idx: dict[int, int],
    emb_norm: np.ndarray,
    item_counts: dict[int, int],
    alpha: float = 0.05,
    lambda_sim: float = 0.0,
    lambda_pop: float = 0.15,
    top_n_anchor: int = 5,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    reranked_items: dict[int, np.ndarray] = {}
    reranked_scores: dict[int, np.ndarray] = {}

    for sid, cand in candidates.items():
        cur_items = np.asarray(cand, dtype=int)
        cur_warm = np.asarray(warm_scores.get(sid, np.zeros(len(cur_items))), dtype=np.float64)

        if cur_items.size == 0:
            reranked_items[sid] = np.array([], dtype=int)
            reranked_scores[sid] = np.array([], dtype=np.float64)
            continue

        q = session_embedding(prompts[sid], item_to_idx, emb_norm, decay=1.0)
        if q is None:
            z_warm = z_score_normalize(cur_warm)
            final = z_warm.copy()
        else:
            idxs = np.array([item_to_idx[int(i)] for i in cur_items], dtype=int)
            cos_sim = emb_norm[idxs] @ q
            pop_penalty = np.array([max(item_counts.get(int(i), 1), 1) for i in cur_items], dtype=np.float64) ** alpha

            z_warm = z_score_normalize(cur_warm)
            z_sim = z_score_normalize(cos_sim)
            z_pop = z_score_normalize(pop_penalty)
            final = z_warm + lambda_sim * z_sim - lambda_pop * z_pop

        if top_n_anchor > 0:
            n_anchor = min(int(top_n_anchor), len(cur_items))
            if n_anchor > 0:
                warm_top_idx = np.argsort(cur_warm)[::-1][:n_anchor]
                anchor_bonus = float(np.max(np.abs(final)) + 1.0) * 2.0
                final[warm_top_idx] += anchor_bonus

        order = np.argsort(final)[::-1]
        reranked_items[sid] = cur_items[order].astype(int)
        reranked_scores[sid] = final[order].astype(np.float64)

    return reranked_items, reranked_scores


def score_cold_candidates(
    candidates: dict[int, np.ndarray],
    prompts: dict[int, np.ndarray],
    item_to_idx: dict[int, int],
    emb_norm: np.ndarray,
    decay: float,
) -> dict[int, np.ndarray]:
    out: dict[int, np.ndarray] = {}
    for sid, cand in candidates.items():
        cur_items = np.asarray(cand, dtype=int)
        if cur_items.size == 0:
            out[sid] = np.array([], dtype=np.float64)
            continue

        q = session_embedding(prompts[sid], item_to_idx, emb_norm, decay=decay)
        if q is None:
            out[sid] = np.zeros(len(cur_items), dtype=np.float64)
            continue

        idxs = np.array([item_to_idx[int(i)] for i in cur_items], dtype=int)
        out[sid] = (emb_norm[idxs] @ q).astype(np.float64)
    return out


def soft_coldbridge_blend(
    warm_scores_arr: np.ndarray,
    cold_scores_arr: np.ndarray,
    session_len: int,
    tau: int = 5,
    steepness: float = 1.0,
) -> np.ndarray:
    x = steepness * (float(tau) - float(session_len))
    p_cold = 1.0 / (1.0 + np.exp(-np.clip(x, -20.0, 20.0)))
    z_warm = z_score_normalize(warm_scores_arr)
    z_cold = z_score_normalize(cold_scores_arr)
    return (1.0 - p_cold) * z_warm + p_cold * z_cold


def build_soft_blend_predictions(
    candidates: dict[int, np.ndarray],
    warm_scores: dict[int, np.ndarray],
    cold_scores: dict[int, np.ndarray],
    prompts: dict[int, np.ndarray],
    tau: int,
    steepness: float,
    top_k: int,
) -> tuple[dict[int, np.ndarray], dict[int, np.ndarray]]:
    out_preds: dict[int, np.ndarray] = {}
    out_scores: dict[int, np.ndarray] = {}
    for sid, cand in candidates.items():
        cur_items = np.asarray(cand, dtype=int)
        cur_warm = np.asarray(warm_scores.get(sid, np.zeros(len(cur_items))), dtype=np.float64)
        cur_cold = np.asarray(cold_scores.get(sid, np.zeros(len(cur_items))), dtype=np.float64)
        if cur_items.size == 0:
            out_preds[sid] = np.array([], dtype=int)
            out_scores[sid] = np.array([], dtype=np.float64)
            continue

        blended = soft_coldbridge_blend(
            warm_scores_arr=cur_warm,
            cold_scores_arr=cur_cold,
            session_len=len(prompts[sid]),
            tau=tau,
            steepness=steepness,
        )
        order = np.argsort(blended)[::-1][:top_k]
        out_preds[sid] = cur_items[order].astype(int)
        out_scores[sid] = blended[order].astype(np.float64)
    return out_preds, out_scores


def is_valid_config(ndcg10: float, hr10: float, coverage: float) -> bool:
    return (hr10 >= HR_FLOOR) and (ndcg10 >= NDCG_FLOOR) and (coverage >= COVERAGE_MIN)


def find_pareto_front(results_df: pd.DataFrame) -> np.ndarray:
    metrics = ["NDCG@10", "HR@10", "CatalogCoverage", "ILD@10"]
    vals = results_df[metrics].to_numpy(dtype=np.float64)
    n = len(vals)
    is_pareto = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if np.all(vals[j] >= vals[i]) and np.any(vals[j] > vals[i]):
                is_pareto[i] = False
                break
    return is_pareto


def cfg_key(alpha: float, lambda_sim: float, lambda_pop: float, top_n_anchor: int, tau: int, steepness: float, decay: float) -> str:
    return (
        f"a={alpha:.4f}|ls={lambda_sim:.4f}|lp={lambda_pop:.4f}|"
        f"an={int(top_n_anchor)}|tau={int(tau)}|st={steepness:.2f}|d={decay:.2f}"
    )

In [ ]:
import itertools

# 1) Data preparation
udata_path, uitem_path = download_ml100k(DATA_DIR, DATASET_URL)
interactions = load_interactions(udata_path)
sessions_df = sessionize(interactions, SESSION_GAP_MINUTES, MIN_SESSION_LEN)
train_df, val_df, test_df = temporal_split(sessions_df, VAL_FRAC, TEST_FRAC)

val_prompts, val_gts, val_lengths = build_cases(val_df)
test_prompts, test_gts, test_lengths = build_cases(test_df)

print("Train sessions:", train_df["SessionId"].nunique())
print("Val sessions:", val_df["SessionId"].nunique())
print("Test sessions:", test_df["SessionId"].nunique())

# 2) Item metadata + BGE embeddings (CPU-safe)
item_metadata = load_item_metadata(uitem_path)
all_item_ids = np.array(sorted(sessions_df["ItemId"].unique().tolist()), dtype=int)
bge_path = EMB_DIR / "item_embeddings_bge_m3.npy"
bge_embeddings = encode_bge_cpu(
    item_ids=all_item_ids,
    item_text_df=item_metadata,
    output_npy=bge_path,
    batch_size=32,
    max_length=256,
)
bge_embeddings = normalize_rows(bge_embeddings.astype(np.float32))

item_to_idx = {int(item): i for i, item in enumerate(all_item_ids.tolist())}
num_items = len(all_item_ids)

# 3) Reduced ID mapping for sequence model
item_to_reduced, reduced_to_item = build_id_maps(all_item_ids)

# 4) Build train examples
x_train, y_train = build_train_examples(train_df, item_to_reduced, SEQ_LEN)
print("Train examples:", x_train.shape, y_train.shape)

# 5) Prepare popularity statistics
item_counts = train_df["ItemId"].value_counts().to_dict()
popular_sorted = [int(i) for i, _ in sorted(item_counts.items(), key=lambda kv: kv[1], reverse=True)]
expected_pop_top10 = popular_sorted[:10]

results_rows: list[dict[str, Any]] = []


def apply_meta(row: dict[str, Any], **meta: Any) -> dict[str, Any]:
    out = dict(row)
    out.update(meta)
    return out


# 6) MostPopular baseline
t0 = time.perf_counter()
popular_preds_test = predict_popular(test_prompts, item_counts, TOP_K)
t1 = time.perf_counter()
row_pop = evaluate_predictions(
    model_name="MostPopular",
    preds=popular_preds_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=None,
    tau=None,
    training_time=0.0,
    inference_time=t1 - t0,
)
row_pop = apply_meta(
    row_pop,
    debias_variant=np.nan,
    cold_decay=np.nan,
    lambda_sim=np.nan,
    lambda_pop=np.nan,
    top_n_anchor=np.nan,
    steepness=np.nan,
    blend_type="baseline",
)
results_rows.append(row_pop)

# 7) SASRec vanilla
sas_model = build_sasrec_like_model(num_items=num_items, seq_len=SEQ_LEN, emb_dim=EMB_DIM, pretrained_item_embeddings=None)
t0 = time.perf_counter()
sas_model.fit(x_train, y_train, epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
sas_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
sas_preds_test = predict_topk(sas_model, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, TOP_K)
sas_inf_time = time.perf_counter() - t0

row_sas = evaluate_predictions(
    model_name="SASRec",
    preds=sas_preds_test,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=None,
    tau=None,
    training_time=sas_train_time,
    inference_time=sas_inf_time,
)
row_sas = apply_meta(
    row_sas,
    debias_variant=np.nan,
    cold_decay=np.nan,
    lambda_sim=np.nan,
    lambda_pop=np.nan,
    top_n_anchor=np.nan,
    steepness=np.nan,
    blend_type="baseline",
)
results_rows.append(row_sas)

# 8) BGE2SASRec (init with PCA-reduced BGE embeddings)
pca = PCA(n_components=EMB_DIM, random_state=SEED)
bge_reduced = pca.fit_transform(bge_embeddings).astype(np.float32)
bge_reduced = normalize_rows(bge_reduced)

bge2sas_model = build_sasrec_like_model(
    num_items=num_items,
    seq_len=SEQ_LEN,
    emb_dim=EMB_DIM,
    pretrained_item_embeddings=bge_reduced,
)
t0 = time.perf_counter()
bge2sas_model.fit(x_train, y_train, epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, verbose=0)
bge2sas_train_time = time.perf_counter() - t0

t0 = time.perf_counter()
base_val_candidates, base_val_scores = predict_topk_with_scores(
    bge2sas_model, val_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
base_test_candidates, base_test_scores = predict_topk_with_scores(
    bge2sas_model, test_prompts, item_to_reduced, reduced_to_item, SEQ_LEN, CANDIDATE_K
)
base_inf_time = time.perf_counter() - t0

bge2sas_test_top = topk_from_candidates_scores(base_test_candidates, base_test_scores, TOP_K)
row_bge2sas = evaluate_predictions(
    model_name="BGE2SASRec",
    preds=bge2sas_test_top,
    gts=test_gts,
    item_counts=item_counts,
    num_items=num_items,
    emb_norm=bge_embeddings,
    item_to_idx=item_to_idx,
    expected_pop_top10=expected_pop_top10,
    alpha=None,
    tau=None,
    training_time=bge2sas_train_time,
    inference_time=base_inf_time,
)
row_bge2sas = apply_meta(
    row_bge2sas,
    debias_variant=np.nan,
    cold_decay=np.nan,
    lambda_sim=np.nan,
    lambda_pop=np.nan,
    top_n_anchor=np.nan,
    steepness=np.nan,
    blend_type="baseline",
)
results_rows.append(row_bge2sas)


def evaluate_v2_config(
    model_name: str,
    *,
    alpha: float,
    lambda_sim: float,
    lambda_pop: float,
    top_n_anchor: int,
    tau: int,
    steepness: float,
    decay: float,
    use_popdebias: bool,
    use_soft_blend: bool,
) -> tuple[dict[str, Any], dict[str, Any], dict[int, np.ndarray], dict[int, np.ndarray]]:
    if use_popdebias:
        val_candidates, val_warm_scores = rerank_popdebias_v2(
            base_val_candidates,
            base_val_scores,
            val_prompts,
            item_to_idx,
            bge_embeddings,
            item_counts,
            alpha=alpha,
            lambda_sim=lambda_sim,
            lambda_pop=lambda_pop,
            top_n_anchor=top_n_anchor,
        )
        test_candidates, test_warm_scores = rerank_popdebias_v2(
            base_test_candidates,
            base_test_scores,
            test_prompts,
            item_to_idx,
            bge_embeddings,
            item_counts,
            alpha=alpha,
            lambda_sim=lambda_sim,
            lambda_pop=lambda_pop,
            top_n_anchor=top_n_anchor,
        )
    else:
        val_candidates, val_warm_scores = base_val_candidates, base_val_scores
        test_candidates, test_warm_scores = base_test_candidates, base_test_scores

    if use_soft_blend:
        val_cold_scores = score_cold_candidates(val_candidates, val_prompts, item_to_idx, bge_embeddings, decay=decay)
        test_cold_scores = score_cold_candidates(test_candidates, test_prompts, item_to_idx, bge_embeddings, decay=decay)
        val_pred, _ = build_soft_blend_predictions(
            val_candidates,
            val_warm_scores,
            val_cold_scores,
            val_prompts,
            tau=tau,
            steepness=steepness,
            top_k=TOP_K,
        )
        test_pred, _ = build_soft_blend_predictions(
            test_candidates,
            test_warm_scores,
            test_cold_scores,
            test_prompts,
            tau=tau,
            steepness=steepness,
            top_k=TOP_K,
        )
        blend_type = "soft"
    else:
        val_pred = topk_from_candidates_scores(val_candidates, val_warm_scores, TOP_K)
        test_pred = topk_from_candidates_scores(test_candidates, test_warm_scores, TOP_K)
        blend_type = "none"

    val_row = evaluate_predictions(
        model_name=f"{model_name}-val",
        preds=val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=alpha if use_popdebias else np.nan,
        tau=tau if use_soft_blend else np.nan,
        training_time=0.0,
        inference_time=0.0,
    )
    test_row = evaluate_predictions(
        model_name=model_name,
        preds=test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=alpha if use_popdebias else np.nan,
        tau=tau if use_soft_blend else np.nan,
        training_time=bge2sas_train_time,
        inference_time=base_inf_time,
    )

    debias_variant = "v2_zblend" if use_popdebias else np.nan
    cold_decay = decay if use_soft_blend else np.nan

    meta = {
        "debias_variant": debias_variant,
        "cold_decay": cold_decay,
        "lambda_sim": lambda_sim if use_popdebias else np.nan,
        "lambda_pop": lambda_pop if use_popdebias else np.nan,
        "top_n_anchor": top_n_anchor if use_popdebias else np.nan,
        "steepness": steepness if use_soft_blend else np.nan,
        "blend_type": blend_type,
    }
    val_row = apply_meta(val_row, **meta)
    test_row = apply_meta(test_row, **meta)
    return val_row, test_row, val_pred, test_pred


# 9) Keep hard ColdBridge baseline sweep (known working)
cold_val_uniform = predict_cold_branch(val_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=1.0)
cold_test_uniform = predict_cold_branch(test_prompts, all_item_ids, item_to_idx, bge_embeddings, TOP_K, decay=1.0)
warm_val_top = topk_from_candidates_scores(base_val_candidates, base_val_scores, TOP_K)
warm_test_top = topk_from_candidates_scores(base_test_candidates, base_test_scores, TOP_K)

tau_val_rows: list[dict[str, Any]] = []
tau_stats = []
hard_tau_to_test: dict[int, dict[str, Any]] = {}
hard_tau_to_preds: dict[int, dict[int, np.ndarray]] = {}

for tau in TAUS:
    val_pred = route_coldbridge(warm_val_top, cold_val_uniform, val_prompts, tau, TOP_K)
    test_pred = route_coldbridge(warm_test_top, cold_test_uniform, test_prompts, tau, TOP_K)

    row_val = evaluate_predictions(
        model_name="ColdBridge-BGE2SASRec-hard-val",
        preds=val_pred,
        gts=val_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=np.nan,
        tau=tau,
        training_time=0.0,
        inference_time=0.0,
    )
    row_val = apply_meta(
        row_val,
        debias_variant=np.nan,
        cold_decay=1.0,
        lambda_sim=np.nan,
        lambda_pop=np.nan,
        top_n_anchor=np.nan,
        steepness=np.nan,
        blend_type="hard",
    )
    tau_val_rows.append(row_val)

    row_test = evaluate_predictions(
        model_name="ColdBridge-BGE2SASRec",
        preds=test_pred,
        gts=test_gts,
        item_counts=item_counts,
        num_items=num_items,
        emb_norm=bge_embeddings,
        item_to_idx=item_to_idx,
        expected_pop_top10=expected_pop_top10,
        alpha=np.nan,
        tau=tau,
        training_time=bge2sas_train_time,
        inference_time=base_inf_time,
    )
    row_test = apply_meta(
        row_test,
        debias_variant=np.nan,
        cold_decay=1.0,
        lambda_sim=np.nan,
        lambda_pop=np.nan,
        top_n_anchor=np.nan,
        steepness=np.nan,
        blend_type="hard",
    )
    results_rows.append(row_test)
    hard_tau_to_test[int(tau)] = row_test
    hard_tau_to_preds[int(tau)] = test_pred

    cold_hr, warm_hr = segmented_hr10(test_pred, test_gts, test_lengths, cold_thr=5)
    tau_stats.append({"tau": float(tau), "cold_hr10": cold_hr, "warm_hr10": warm_hr})

best_hard_tau_row = choose_best_joint(tau_val_rows)
best_hard_tau = int(best_hard_tau_row["tau"])
best_hard_test_row = dict(hard_tau_to_test[best_hard_tau])
best_hard_test_pred = hard_tau_to_preds[best_hard_tau]
print("Best hard ColdBridge tau:", best_hard_tau)

# 10) Required ablation table (v2)
ablation_records: list[dict[str, Any]] = []

# Step 0: Base
ablation_records.append(
    {
        "step": 0,
        "model": "BGE2SASRec (base)",
        "components": "None",
        "NDCG@10": float(row_bge2sas["NDCG@10"]),
        "HR@10": float(row_bge2sas["HR@10"]),
    }
)

# Step 1: + SoftColdBridge
val_s1, test_s1, _, _ = evaluate_v2_config(
    "SoftColdBridge-BGE2SASRec-v2",
    alpha=0.0,
    lambda_sim=0.0,
    lambda_pop=0.0,
    top_n_anchor=0,
    tau=5,
    steepness=1.0,
    decay=1.0,
    use_popdebias=False,
    use_soft_blend=True,
)
results_rows.append(test_s1)
ablation_records.append(
    {
        "step": 1,
        "model": "SoftColdBridge-BGE2SASRec-v2",
        "components": "Soft gate (tau=5, steep=1.0)",
        "NDCG@10": float(test_s1["NDCG@10"]),
        "HR@10": float(test_s1["HR@10"]),
    }
)

# Step 2: + SafePopDebias (without cold blending)
val_s2, test_s2, _, _ = evaluate_v2_config(
    "SafePopDebias-BGE2SASRec-v2",
    alpha=0.05,
    lambda_sim=0.0,
    lambda_pop=0.15,
    top_n_anchor=5,
    tau=5,
    steepness=1.0,
    decay=1.0,
    use_popdebias=True,
    use_soft_blend=False,
)
results_rows.append(test_s2)
ablation_records.append(
    {
        "step": 2,
        "model": "SafePopDebias-BGE2SASRec-v2",
        "components": "z-blend pop debias only",
        "NDCG@10": float(test_s2["NDCG@10"]),
        "HR@10": float(test_s2["HR@10"]),
    }
)

# Step 3: + Both
val_s3, test_s3, _, _ = evaluate_v2_config(
    "PopDebias-ColdBridge-BGE2SASRec-v2",
    alpha=0.05,
    lambda_sim=0.0,
    lambda_pop=0.15,
    top_n_anchor=5,
    tau=5,
    steepness=1.0,
    decay=1.0,
    use_popdebias=True,
    use_soft_blend=True,
)
results_rows.append(test_s3)
ablation_records.append(
    {
        "step": 3,
        "model": "PopDebias-ColdBridge-BGE2SASRec-v2",
        "components": "SafePopDebias + SoftColdBridge",
        "NDCG@10": float(test_s3["NDCG@10"]),
        "HR@10": float(test_s3["HR@10"]),
    }
)

# Step 4: + WeightedCold
def weighted_decay_for_step4() -> float:
    return 0.8

val_s4, test_s4, _, _ = evaluate_v2_config(
    "PopDebias-ColdBridge-WeightedCold-v2",
    alpha=0.05,
    lambda_sim=0.0,
    lambda_pop=0.15,
    top_n_anchor=5,
    tau=5,
    steepness=1.0,
    decay=weighted_decay_for_step4(),
    use_popdebias=True,
    use_soft_blend=True,
)
results_rows.append(test_s4)
ablation_records.append(
    {
        "step": 4,
        "model": "PopDebias-ColdBridge-WeightedCold-v2",
        "components": "SafePopDebias + SoftColdBridge + decay=0.8",
        "NDCG@10": float(test_s4["NDCG@10"]),
        "HR@10": float(test_s4["HR@10"]),
    }
)

# 11) Constrained v2 hyperparameter sweep
ALPHA_GRID = [0.01, 0.05, 0.1, 0.15, 0.2]
LAMBDA_POP_GRID = [0.05, 0.1, 0.15, 0.2]
LAMBDA_SIM_GRID = [0.0, 0.05, 0.1]
TAU_GRID = [3, 5, 7, 10, 15]
STEEPNESS_GRID = [0.5, 1.0, 2.0]
DECAY_GRID = [0.7, 0.8, 0.9, 1.0]
TOP_N_ANCHOR_GRID = [3, 5]

valid_val_rows: list[dict[str, Any]] = []
valid_test_rows: list[dict[str, Any]] = []
config_to_test_row: dict[str, dict[str, Any]] = {}

for alpha, lambda_pop, lambda_sim, tau, steepness, decay, top_n_anchor in itertools.product(
    ALPHA_GRID,
    LAMBDA_POP_GRID,
    LAMBDA_SIM_GRID,
    TAU_GRID,
    STEEPNESS_GRID,
    DECAY_GRID,
    TOP_N_ANCHOR_GRID,
):
    val_row, test_row, _, _ = evaluate_v2_config(
        "FULL_SYSTEM_v2",
        alpha=alpha,
        lambda_sim=lambda_sim,
        lambda_pop=lambda_pop,
        top_n_anchor=top_n_anchor,
        tau=tau,
        steepness=steepness,
        decay=decay,
        use_popdebias=True,
        use_soft_blend=True,
    )

    ndcg10 = float(val_row["NDCG@10"])
    hr10 = float(val_row["HR@10"])
    cov = float(val_row["CatalogCoverage"])
    ild = float(val_row["ILD@10"])
    joint_raw = 0.5 * ndcg10 + 0.3 * hr10 + 0.2 * cov

    print(
        f"[CONFIG] model=FULL_SYSTEM_v2 alpha={alpha} tau={tau} lambda_pop={lambda_pop} "
        f"lambda_sim={lambda_sim} top_n_anchor={top_n_anchor} steepness={steepness} decay={decay}"
    )
    print(f"[RESULT] NDCG@10={ndcg10:.6f} HR@10={hr10:.6f} Coverage={cov:.6f} ILD={ild:.6f}")

    if not is_valid_config(ndcg10, hr10, cov):
        reason_parts = []
        if hr10 < HR_FLOOR:
            reason_parts.append(f"HR@10<{HR_FLOOR}")
        if ndcg10 < NDCG_FLOOR:
            reason_parts.append(f"NDCG@10<{NDCG_FLOOR}")
        if cov < COVERAGE_MIN:
            reason_parts.append(f"Coverage<{COVERAGE_MIN}")
        print(f"[STATUS] REJECTED ({'; '.join(reason_parts)})")
        print(f"[JOINT ] joint_score={joint_raw:.6f}")
        continue

    print("[STATUS] VALID")
    print(f"[JOINT ] joint_score={joint_raw:.6f}")

    key = cfg_key(alpha, lambda_sim, lambda_pop, top_n_anchor, tau, steepness, decay)
    val_row["config_key"] = key
    val_row["joint_raw"] = joint_raw
    test_row["config_key"] = key

    valid_val_rows.append(val_row)
    valid_test_rows.append(test_row)
    config_to_test_row[key] = dict(test_row)
    results_rows.append(dict(test_row))

# Choose best v2 config by normalized joint on VALID validation rows
best_v2_row: dict[str, Any]
if valid_val_rows:
    valid_val_df = pd.DataFrame(valid_val_rows)
    ndcg_best = float(valid_val_df["NDCG@10"].max())
    hr_best = float(valid_val_df["HR@10"].max())
    cov_best = float(valid_val_df["CatalogCoverage"].max())

    valid_val_df["joint_score"] = (
        0.5 * (valid_val_df["NDCG@10"] / max(ndcg_best, 1e-9))
        + 0.3 * (valid_val_df["HR@10"] / max(hr_best, 1e-9))
        + 0.2 * (valid_val_df["CatalogCoverage"] / max(cov_best, 1e-9))
    )
    best_idx = int(valid_val_df["joint_score"].idxmax())
    best_key = str(valid_val_df.loc[best_idx, "config_key"])

    best_v2_row = dict(config_to_test_row[best_key])
    best_v2_row["model_name"] = "BEST_FULL_SYSTEM_v2"
    best_v2_row["joint_score"] = float(valid_val_df.loc[best_idx, "joint_score"])
    results_rows.append(best_v2_row)
else:
    best_v2_row = dict(best_hard_test_row)
    best_v2_row["model_name"] = "BEST_FULL_SYSTEM_v2_FALLBACK_HARD"
    best_v2_row["joint_score"] = np.nan
    results_rows.append(best_v2_row)

# 12) Pareto frontier on valid v2 candidates
pareto_input_df = pd.DataFrame(valid_test_rows)
if not pareto_input_df.empty:
    pareto_mask = find_pareto_front(pareto_input_df)
    pareto_df = pareto_input_df.loc[pareto_mask].copy()
else:
    pareto_df = pd.DataFrame(columns=[
        "model_name",
        "alpha",
        "tau",
        "NDCG@10",
        "NDCG@20",
        "HR@10",
        "HR@20",
        "LongTail_HR@10",
        "CatalogCoverage",
        "Serendipity",
        "ILD@10",
        "training_time_sec",
        "inference_time_sec",
        "debias_variant",
        "cold_decay",
        "lambda_sim",
        "lambda_pop",
        "top_n_anchor",
        "steepness",
        "blend_type",
    ])

pareto_csv = RESULTS_DIR / "pareto_front_v2.csv"
pareto_df.to_csv(pareto_csv, index=False)

# 13) Save ablation table
ablation_path = RESULTS_DIR / "ablation_v2.md"
ablation_lines = [
    "# Ablation v2",
    "",
    "| Step | Model | Components Active | NDCG@10 | HR@10 |",
    "|---|---|---|---:|---:|",
]
for rec in sorted(ablation_records, key=lambda x: x["step"]):
    ablation_lines.append(
        f"| {rec['step']} | {rec['model']} | {rec['components']} | {rec['NDCG@10']:.6f} | {rec['HR@10']:.6f} |"
    )
ablation_path.write_text("\n".join(ablation_lines) + "\n", encoding="utf-8")

# 14) Append to results CSV (keep legacy rows)
results_csv = RESULTS_DIR / "full_results.csv"
schema_cols = [
    "model_name",
    "alpha",
    "tau",
    "NDCG@10",
    "NDCG@20",
    "HR@10",
    "HR@20",
    "LongTail_HR@10",
    "CatalogCoverage",
    "Serendipity",
    "ILD@10",
    "training_time_sec",
    "inference_time_sec",
    "debias_variant",
    "cold_decay",
    "lambda_sim",
    "lambda_pop",
    "top_n_anchor",
    "steepness",
    "blend_type",
]

new_results_df = pd.DataFrame(results_rows)
for c in schema_cols:
    if c not in new_results_df.columns:
        new_results_df[c] = np.nan
new_results_df = new_results_df[schema_cols]

if results_csv.exists():
    old_df = pd.read_csv(results_csv)
    for c in schema_cols:
        if c not in old_df.columns:
            old_df[c] = np.nan
    old_df = old_df[schema_cols]
    merged_df = pd.concat([old_df, new_results_df], ignore_index=True)
else:
    merged_df = new_results_df.copy()

merged_df.to_csv(results_csv, index=False)

# 15) Save best_model.txt (v2 format vs core targets)
best_txt = RESULTS_DIR / "best_model.txt"


def pct_delta(cur: float, ref: float) -> float:
    if ref == 0:
        return float("nan")
    return ((cur - ref) / ref) * 100.0


best_model_name = str(best_v2_row.get("model_name", "UNKNOWN"))
best_ndcg = float(best_v2_row.get("NDCG@10", 0.0))
best_hr = float(best_v2_row.get("HR@10", 0.0))
best_cov = float(best_v2_row.get("CatalogCoverage", 0.0))
best_lt = float(best_v2_row.get("LongTail_HR@10", 0.0))
best_ild = float(best_v2_row.get("ILD@10", 0.0))

alpha_cfg = best_v2_row.get("alpha", np.nan)
tau_cfg = best_v2_row.get("tau", np.nan)
lambda_pop_cfg = best_v2_row.get("lambda_pop", np.nan)
steepness_cfg = best_v2_row.get("steepness", np.nan)
decay_cfg = best_v2_row.get("cold_decay", np.nan)

llm2 = CORE_TARGETS["LLM2SASRec"]
bert4 = CORE_TARGETS["BERT4Rec"]
core_sas = CORE_TARGETS["SASRec"]

lines = [
    f"BEST MODEL v2: {best_model_name}",
    f"Config: alpha={alpha_cfg}, tau={tau_cfg}, lambda_pop={lambda_pop_cfg}, steepness={steepness_cfg}, decay={decay_cfg}",
    "",
    f"NDCG@10:        {best_ndcg:.6f}  (vs Core LLM2SASRec: {llm2['NDCG@10']:.5f}, delta: {pct_delta(best_ndcg, llm2['NDCG@10']):+.2f}%)",
    f"HR@10:          {best_hr:.6f}  (vs Core LLM2SASRec: {llm2['HR@10']:.5f}, delta: {pct_delta(best_hr, llm2['HR@10']):+.2f}%)",
    f"LongTail_HR@10: {best_lt:.6f}",
    f"CatalogCoverage:{best_cov:.6f} (vs Core LLM2SASRec: {llm2['CatalogCoverage']:.5f}, delta: {pct_delta(best_cov, llm2['CatalogCoverage']):+.2f}%)",
    f"ILD@10:         {best_ild:.6f}",
    "",
    f"BEATS Core LLM2SASRec on NDCG@10: {'YES' if best_ndcg > llm2['NDCG@10'] else 'NO'}",
    f"BEATS Core BERT4Rec    on NDCG@10: {'YES' if best_ndcg > bert4['NDCG@10'] else 'NO'}",
    f"BEATS Core SASRec      on NDCG@10: {'YES' if best_ndcg > core_sas['NDCG@10'] else 'NO'}",
]

best_txt.write_text("\n".join(lines) + "\n", encoding="utf-8")

# 16) Figures (hard ColdBridge tau + ablation summary)
if tau_stats:
    tau_df = pd.DataFrame(tau_stats).sort_values("tau")
    plt.figure(figsize=(7, 5))
    plt.plot(tau_df["tau"], tau_df["cold_hr10"], marker="o", label="Cold HR@10")
    plt.plot(tau_df["tau"], tau_df["warm_hr10"], marker="s", label="Warm HR@10")
    plt.xlabel("tau")
    plt.ylabel("HR@10")
    plt.title("Hard ColdBridge tau sweep")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / "tau_sweep_movielens_100k.png", dpi=180)
    plt.close()

abl_df = pd.DataFrame(ablation_records).sort_values("step")
plt.figure(figsize=(8, 5))
plt.plot(abl_df["step"], abl_df["NDCG@10"], marker="o", label="NDCG@10")
plt.plot(abl_df["step"], abl_df["HR@10"], marker="s", label="HR@10")
plt.xticks(abl_df["step"].tolist())
plt.xlabel("Ablation step")
plt.ylabel("Metric")
plt.title("Ablation v2 progression")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "ablation_v2_progress.png", dpi=180)
plt.close()

base_cold, base_warm = segmented_hr10(bge2sas_test_top, test_gts, test_lengths, cold_thr=5)
full_cold, full_warm = segmented_hr10(best_hard_test_pred, test_gts, test_lengths, cold_thr=5)
x = np.arange(2)
w = 0.35
plt.figure(figsize=(7, 5))
plt.bar(x - w / 2, [base_cold, base_warm], w, label="BGE2SASRec")
plt.bar(x + w / 2, [full_cold, full_warm], w, label="Hard ColdBridge best")
plt.xticks(x, ["Cold", "Warm"])
plt.ylabel("HR@10")
plt.title("Cold/Warm HR@10 gap")
plt.grid(axis="y", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "cold_warm_gap_movielens_100k.png", dpi=180)
plt.close()

print("Done.")
print("Results CSV:", results_csv)
print("Best model:", best_txt)
print("Ablation v2:", ablation_path)
print("Pareto front:", pareto_csv)
print("Figures dir:", FIG_DIR)
display(merged_df.sort_values(["NDCG@10", "HR@10"], ascending=False).head(30))

Train sessions: 1598
Val sessions: 228
Test sessions: 457


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Train examples: (69582, 20) (69582,)


I0000 00:00:1776690186.020090      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1776690189.691606     761 service.cc:152] XLA service 0x78eb840036e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776690189.691652     761 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1776690190.255541     761 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776690193.458428     761 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Best alpha: 0.1
Best tau: 15
Best variant: log_norm alpha: 0.1
Best decay: 1.0
Done.
Results CSV: /kaggle/working/llmseqrec_ml100k_single_notebook/results/movielens_100k/full_results.csv
Best model: /kaggle/working/llmseqrec_ml100k_single_notebook/results/movielens_100k/best_model.txt
Figures dir: /kaggle/working/llmseqrec_ml100k_single_notebook/results/figures


,model_name,alpha,tau,NDCG@10,NDCG@20,HR@10,HR@20,LongTail_HR@10,CatalogCoverage,Serendipity,ILD@10,training_time_sec,inference_time_sec,debias_variant,cold_decay
11,ColdBridge-BGE2SASRec,NaN,15.0,0.058122,0.071388,0.094092,0.148796,0.094092,0.528886,0.091904,0.173683,14.476991,1.698569,NaN,NaN
9,ColdBridge-BGE2SASRec,NaN,5.0,0.057809,0.070646,0.094092,0.146608,0.094092,0.515783,0.091904,0.187545,14.476991,1.698569,NaN,NaN
10,ColdBridge-BGE2SASRec,NaN,10.0,0.057325,0.070592,0.091904,0.146608,0.091904,0.527099,0.089716,0.178454,14.476991,1.698569,NaN,NaN
8,ColdBridge-BGE2SASRec,NaN,3.0,0.054375,0.065616,0.091904,0.137856,0.091904,0.506849,0.087527,0.192811,14.476991,1.698569,NaN,NaN
7,ColdBridge-BGE2SASRec,NaN,2.0,0.052916,0.064265,0.091904,0.137856,0.091904,0.492555,0.087527,0.198264,14.476991,1.698569,NaN,NaN
1,SASRec,NaN,NaN,0.047220,0.055436,0.100656,0.133479,0.100656,0.301370,0.089716,0.213208,15.279092,0.939186,NaN,NaN
2,BGE2SASRec,NaN,NaN,0.046117,0.058570,0.085339,0.135667,0.085339,0.317451,0.076586,0.213129,14.476991,1.698569,NaN,NaN
16,PopDebias-ColdBridge-rank,0.1,15.0,0.031374,0.043817,0.052516,0.102845,0.052516,0.372245,0.039387,0.174766,14.476991,1.698569,rank,NaN
17,PopDebias-ColdBridge-rank,0.3,15.0,0.031129,0.042968,0.050328,0.098468,0.050328,0.379988,0.037199,0.178092,14.476991,1.698569,rank,NaN
18,PopDebias-ColdBridge-rank,0.5,15.0,0.031129,0.043381,0.050328,0.100656,0.050328,0.381179,0.037199,0.178709,14.476991,1.698569,rank,NaN


In [ ]:
# Optional: quick artifact preview after the main run cell
results_csv = RESULTS_DIR / "full_results.csv"
best_txt = RESULTS_DIR / "best_model.txt"
figures_dir = FIG_DIR
ablation_md = RESULTS_DIR / "ablation_v2.md"
pareto_csv = RESULTS_DIR / "pareto_front_v2.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.head(20))
else:
    print("Results file not found:", results_csv)

if best_txt.exists():
    print("=== best_model.txt ===")
    print(best_txt.read_text(encoding="utf-8"))
else:
    print("Best-model file not found:", best_txt)

if ablation_md.exists():
    print("=== ablation_v2.md ===")
    print(ablation_md.read_text(encoding="utf-8"))
else:
    print("Ablation file not found:", ablation_md)

if pareto_csv.exists():
    print("=== pareto_front_v2.csv (head) ===")
    display(pd.read_csv(pareto_csv).head(20))
else:
    print("Pareto file not found:", pareto_csv)

if figures_dir.exists():
    print("Figures:")
    for p in sorted(figures_dir.glob("*.png")):
        print("-", p.name)

,model_name,alpha,tau,NDCG@10,NDCG@20,HR@10,HR@20,LongTail_HR@10,CatalogCoverage,Serendipity,ILD@10,training_time_sec,inference_time_sec,debias_variant,cold_decay
0,ColdBridge-BGE2SASRec,NaN,15.0,0.058122,0.071388,0.094092,0.148796,0.094092,0.528886,0.091904,0.173683,14.476991,1.698569,NaN,NaN
1,ColdBridge-BGE2SASRec,NaN,5.0,0.057809,0.070646,0.094092,0.146608,0.094092,0.515783,0.091904,0.187545,14.476991,1.698569,NaN,NaN
2,ColdBridge-BGE2SASRec,NaN,10.0,0.057325,0.070592,0.091904,0.146608,0.091904,0.527099,0.089716,0.178454,14.476991,1.698569,NaN,NaN
3,ColdBridge-BGE2SASRec,NaN,3.0,0.054375,0.065616,0.091904,0.137856,0.091904,0.506849,0.087527,0.192811,14.476991,1.698569,NaN,NaN
4,ColdBridge-BGE2SASRec,NaN,2.0,0.052916,0.064265,0.091904,0.137856,0.091904,0.492555,0.087527,0.198264,14.476991,1.698569,NaN,NaN
5,SASRec,NaN,NaN,0.047220,0.055436,0.100656,0.133479,0.100656,0.301370,0.089716,0.213208,15.279092,0.939186,NaN,NaN
6,BGE2SASRec,NaN,NaN,0.046117,0.058570,0.085339,0.135667,0.085339,0.317451,0.076586,0.213129,14.476991,1.698569,NaN,NaN
7,PopDebias-ColdBridge-rank,0.1,15.0,0.031374,0.043817,0.052516,0.102845,0.052516,0.372245,0.039387,0.174766,14.476991,1.698569,rank,NaN
8,PopDebias-ColdBridge-rank,0.3,15.0,0.031129,0.042968,0.050328,0.098468,0.050328,0.379988,0.037199,0.178092,14.476991,1.698569,rank,NaN
9,PopDebias-ColdBridge-rank,0.5,15.0,0.031129,0.043381,0.050328,0.100656,0.050328,0.381179,0.037199,0.178709,14.476991,1.698569,rank,NaN


=== best_model.txt ===
BEST MODEL: ColdBridge-BGE2SASRec
Best Alpha: nan
Best Tau:   15.0
NDCG@10:    0.058122 (vs baseline: 0.046117, delta: +26.03%)
HR@10:      0.094092
LT-HR@10:   0.094092
ILD@10:     0.173683

Figures:
- alpha_sweep_movielens_100k.png
- cold_warm_gap_movielens_100k.png
- tau_sweep_movielens_100k.png


Default is now `RUN_MODE = "fast"` in Cell 6, which applies a reduced sweep grid, a capped trial count, and `COVERAGE_MIN = 0.30`.

Set `RUN_MODE = "full"` in Cell 6 to restore the exhaustive sweep and the stricter `COVERAGE_MIN = 0.35`.

If Kaggle runtime is still tight, lower `NUM_EPOCHS` (for example from `4` to `3`) and reduce `CANDIDATE_K` (for example from `300` to `200`) before re-running.